# Episode 11: Linear Regression
### Atrangi AI — ML Playlist

This notebook has two parts:

1. **Part 1** — The exact simple example used in the video (Chai Sales vs Temperature)
2. **Part 2 onwards** — Extra material for people who want to go deeper than the video (real datasets, multiple features, polynomial regression, feature scaling)

If you're just following along with the video, Part 1 is all you need. Everything after that is bonus.


---
## Part 1: The Video Example — Chai Sales vs Temperature

Same data, same code, same order as the video.


### Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("Libraries imported!")

### Cell 2: Create the Data

In [ ]:
data = {
    "Temperature": [15, 18, 20, 22, 25, 28, 30, 32, 35, 38],
    "Chai_Sales":  [90, 85, 78, 70, 60, 48, 40, 32, 20, 10]
}

df = pd.DataFrame(data)
df

### Cell 3: Visualize the Data (Scatter Plot)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Temperature"], df["Chai_Sales"], color="darkorange", s=80)
plt.xlabel("Temperature (°C)")
plt.ylabel("Chai Sales (cups)")
plt.title("Temperature vs Chai Sales")
plt.grid(True, alpha=0.3)
plt.show()

### Cell 4: Split into Features (X) and Target (y)

In [ ]:
X = df[["Temperature"]]   # Feature (must be 2D)
y = df["Chai_Sales"]      # Target

print("X shape:", X.shape)
print("y shape:", y.shape)

### Cell 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### Cell 6: Create and Train the Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained!")
print("Slope (m):", model.coef_[0])
print("Intercept (b):", model.intercept_)

**What this means:**

`Chai_Sales = (m × Temperature) + b`

The slope is negative — makes sense, as temperature goes up, chai sales go down. The model has learned this pattern purely from the data.


### Cell 7: Make Predictions on Test Data

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Temperature": X_test["Temperature"].values,
    "Actual Sales": y_test.values,
    "Predicted Sales": y_pred.round(1)
})
comparison

### Cell 8: Evaluate the Model

In [ ]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", round(mse, 2))
print("R² Score:", round(r2, 3))

**Reading these numbers:**

- **MSE** — average squared error between actual and predicted values. Lower is better. It's in squared units, so it's more useful for comparing models than for judging one model in isolation.
- **R² Score** — how much of the variation in Chai Sales is explained by Temperature. Ranges from 0 to 1 (can go negative for a bad model). Closer to 1 = better fit.


### Cell 9: Visualize the Best-Fit Line

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Temperature"], df["Chai_Sales"], color="darkorange", s=80, label="Actual Data")

# Draw the line across the full temperature range
x_line = np.linspace(df["Temperature"].min(), df["Temperature"].max(), 100).reshape(-1, 1)
y_line = model.predict(pd.DataFrame(x_line, columns=["Temperature"]))

plt.plot(x_line, y_line, color="blue", linewidth=2, label="Best Fit Line")
plt.xlabel("Temperature (°C)")
plt.ylabel("Chai Sales (cups)")
plt.title("Linear Regression: Best Fit Line")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Cell 10: Predict on New, Unseen Data

In [ ]:
# Note: model was trained on a DataFrame with a named column "Temperature".
# Predicting with a plain array like np.array([[17]]) throws a UserWarning
# because it has no feature names. Fix: wrap it in a DataFrame with the same column name.

new_temp = pd.DataFrame([[17]], columns=["Temperature"])
predicted_sales = model.predict(new_temp)

print(f"Predicted Chai Sales at 17°C: {predicted_sales[0]:.1f} cups")

> This is the exact fix discussed in the video — always match the input shape (and column names, if trained on a DataFrame) to what the model was trained on.


---
## Part 2: Multiple Linear Regression — House Price Prediction

The video used **one** feature (Temperature). Real problems usually have **many** features. Let's predict house price using Size, Bedrooms, and Age.


### Cell 11: Create a Multi-Feature Dataset

In [ ]:
np.random.seed(42)
n = 100

size_sqft = np.random.randint(500, 3500, n)
bedrooms = np.random.randint(1, 6, n)
age_years = np.random.randint(0, 40, n)

# Price formula with some noise, just to simulate a realistic relationship
price = (size_sqft * 150) + (bedrooms * 10000) - (age_years * 800) + np.random.normal(0, 15000, n)

house_df = pd.DataFrame({
    "Size_sqft": size_sqft,
    "Bedrooms": bedrooms,
    "Age_years": age_years,
    "Price": price.round(0)
})

house_df.head()

### Cell 12: Check Correlations

In [ ]:
correlation = house_df.corr()["Price"].sort_values(ascending=False)
print(correlation)

plt.figure(figsize=(6, 4))
plt.bar(correlation.index[1:], correlation.values[1:], color=["green" if v > 0 else "red" for v in correlation.values[1:]])
plt.title("Correlation of Each Feature with Price")
plt.ylabel("Correlation")
plt.axhline(0, color="black", linewidth=0.8)
plt.show()

**Why this matters:** Size is strongly positively correlated with price (bigger house = higher price). Age is negatively correlated (older house = lower price). This is a sanity check before training — if a feature has near-zero correlation, it might not help the model much.


### Cell 13: Train Multiple Linear Regression

In [ ]:
X_house = house_df[["Size_sqft", "Bedrooms", "Age_years"]]
y_house = house_df["Price"]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

house_model = LinearRegression()
house_model.fit(X_train_h, y_train_h)

print("Intercept (b):", round(house_model.intercept_, 2))
print("\nFeature weights:")
for feature, coef in zip(X_house.columns, house_model.coef_):
    print(f"  {feature}: {coef:.2f}")

### Cell 14: Visualize Feature Weights

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(X_house.columns, house_model.coef_, color="steelblue")
plt.xlabel("Weight (impact on price)")
plt.title("How Much Each Feature Influences Price")
plt.axvline(0, color="black", linewidth=0.8)
plt.grid(True, alpha=0.3, axis="x")
plt.show()

**Reading this:** each weight tells you how much price changes for a 1-unit increase in that feature, holding the others constant. Size has the biggest raw weight, but remember — Size is measured in sqft (large numbers) while Bedrooms is measured in count (small numbers), so comparing raw weights directly across features can be misleading. That's exactly what Part 5 (Feature Scaling) below fixes.


### Cell 15: Evaluate the House Price Model

In [ ]:
y_pred_h = house_model.predict(X_test_h)

mse_h = mean_squared_error(y_test_h, y_pred_h)
r2_h = r2_score(y_test_h, y_pred_h)

print("MSE:", round(mse_h, 2))
print("RMSE:", round(np.sqrt(mse_h), 2))
print("R² Score:", round(r2_h, 3))

**RMSE (Root Mean Squared Error)** is just the square root of MSE — it brings the error back into the original units (rupees, in this case), which makes it easier to interpret than MSE alone.


---
## Part 3: Visualizing Model Errors Properly

With one feature, you can draw a 2D best-fit line. With multiple features, you can't draw the line anymore — but you can still visualize how good the model is.


### Cell 16: Predicted vs Actual Plot

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_h, y_pred_h, alpha=0.6, color="teal")

# Perfect prediction line (y = x)
min_val = min(y_test_h.min(), y_pred_h.min())
max_val = max(y_test_h.max(), y_pred_h.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Perfect Prediction")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Predicted vs Actual House Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**How to read this:** the closer the points sit to the red dashed line, the better the model. Points far above the line = model over-predicted. Points far below = model under-predicted.


### Cell 17: Residual Plot

In [ ]:
residuals = y_test_h - y_pred_h

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_h, residuals, alpha=0.6, color="purple")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()

**What a healthy residual plot looks like:** points scattered randomly around the zero line, with no visible pattern. If you see a curve or a funnel shape, it usually means Linear Regression is the wrong model for this data — which brings us to Part 4.


---
## Part 4: When a Straight Line Isn't Enough — Polynomial Regression

Linear Regression assumes a straight-line relationship. Some real relationships are curved. Example: Speed vs Braking Distance — braking distance doesn't grow linearly with speed, it grows much faster.


### Cell 18: Create a Curved Relationship Dataset

In [ ]:
np.random.seed(1)
speed = np.linspace(10, 120, 40)
braking_distance = 0.05 * speed**2 + np.random.normal(0, 15, 40)  # quadratic relationship

speed_df = pd.DataFrame({"Speed": speed, "Braking_Distance": braking_distance})

plt.figure(figsize=(8, 5))
plt.scatter(speed_df["Speed"], speed_df["Braking_Distance"], color="crimson")
plt.xlabel("Speed (km/h)")
plt.ylabel("Braking Distance (m)")
plt.title("Speed vs Braking Distance (Curved Relationship)")
plt.grid(True, alpha=0.3)
plt.show()

### Cell 19: Fit a Plain Linear Regression (Watch it Fail)

In [ ]:
X_speed = speed_df[["Speed"]]
y_speed = speed_df["Braking_Distance"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_speed, y_speed, test_size=0.2, random_state=42)

linear_model = LinearRegression()
linear_model.fit(X_train_s, y_train_s)
y_pred_linear = linear_model.predict(X_test_s)

r2_linear = r2_score(y_test_s, y_pred_linear)
print("Plain Linear Regression R²:", round(r2_linear, 3))

### Cell 20: Fit Polynomial Regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_s)
X_test_poly = poly.transform(X_test_s)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train_s)
y_pred_poly = poly_model.predict(X_test_poly)

r2_poly = r2_score(y_test_s, y_pred_poly)
print("Polynomial Regression (degree 2) R²:", round(r2_poly, 3))

**Important:** Polynomial Regression is still technically "linear regression" under the hood — it's linear in the *coefficients*, just fed curved versions of the input (x, x²). That's why the same `LinearRegression()` class is reused — only the input features change.


### Cell 21: Compare Both Fits Visually

In [ ]:
x_range = np.linspace(speed_df["Speed"].min(), speed_df["Speed"].max(), 100).reshape(-1, 1)
x_range_df = pd.DataFrame(x_range, columns=["Speed"])

y_line_linear = linear_model.predict(x_range_df)
y_line_poly = poly_model.predict(poly.transform(x_range_df))

plt.figure(figsize=(9, 6))
plt.scatter(speed_df["Speed"], speed_df["Braking_Distance"], color="crimson", alpha=0.6, label="Actual Data")
plt.plot(x_range, y_line_linear, color="blue", linewidth=2, label=f"Linear Fit (R²={r2_linear:.2f})")
plt.plot(x_range, y_line_poly, color="green", linewidth=2, label=f"Polynomial Fit (R²={r2_poly:.2f})")
plt.xlabel("Speed (km/h)")
plt.ylabel("Braking Distance (m)")
plt.title("Linear vs Polynomial Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Takeaway:** always look at your data before picking a model. A residual plot with a curved pattern (Part 3) is the warning sign that tells you to try this.


---
## Part 5: Feature Scaling — Making Weights Comparable

Back in Part 2, we saw that Size (large numbers, sqft) had a much bigger raw weight than Bedrooms (small numbers, 1-5). That doesn't necessarily mean Size matters more — it might just be the units. Feature Scaling fixes this.


### Cell 22: Scale the House Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_h)
X_test_scaled = scaler.transform(X_test_h)

scaled_model = LinearRegression()
scaled_model.fit(X_train_scaled, y_train_h)

print("Weights BEFORE scaling:")
for feature, coef in zip(X_house.columns, house_model.coef_):
    print(f"  {feature}: {coef:.2f}")

print("\nWeights AFTER scaling:")
for feature, coef in zip(X_house.columns, scaled_model.coef_):
    print(f"  {feature}: {coef:.2f}")

**What changed:** after scaling, every feature is on the same footing (mean 0, standard deviation 1). Now the weight sizes actually reflect how much each feature matters to the prediction — not just what units it happens to be measured in.

Scaling doesn't change the model's predictions or R² score for plain Linear Regression — it only changes how interpretable the weights are. (For other algorithms like Logistic Regression or Neural Networks, scaling *does* affect performance — but that's a story for another episode.)


### Cell 23: Confirm R² is Unchanged by Scaling

In [ ]:
y_pred_scaled = scaled_model.predict(X_test_scaled)
r2_scaled = r2_score(y_test_h, y_pred_scaled)

print("R² without scaling:", round(r2_h, 3))
print("R² with scaling:   ", round(r2_scaled, 3))

---
## Part 6: A Real, Large-Scale Dataset — California Housing

Everything above used made-up data. Let's finish with a real dataset that has thousands of rows and see how Linear Regression holds up.


### Cell 24: Load the California Housing Dataset

In [ ]:
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing(as_frame=True)
cal_df = california.frame

print("Shape:", cal_df.shape)
cal_df.head()

### Cell 25: Train on the Real Dataset

In [ ]:
X_cal = cal_df.drop(columns=["MedHouseVal"])
y_cal = cal_df["MedHouseVal"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cal, y_cal, test_size=0.2, random_state=42)

cal_model = LinearRegression()
cal_model.fit(X_train_c, y_train_c)

y_pred_c = cal_model.predict(X_test_c)

print("MSE:", round(mean_squared_error(y_test_c, y_pred_c), 3))
print("R² Score:", round(r2_score(y_test_c, y_pred_c), 3))

### Cell 26: Which Features Matter Most Here?

In [ ]:
cal_weights = pd.Series(cal_model.coef_, index=X_cal.columns).sort_values()

plt.figure(figsize=(8, 5))
plt.barh(cal_weights.index, cal_weights.values, color="darkslateblue")
plt.title("Feature Weights — California Housing")
plt.xlabel("Weight")
plt.axvline(0, color="black", linewidth=0.8)
plt.grid(True, alpha=0.3, axis="x")
plt.show()

**Real-world note:** R² around 0.6 is common for real housing data with only Linear Regression — real relationships are messier than our synthetic examples. This is exactly why more powerful models (Decision Trees, Random Forest — which you've already seen in Episodes 10 and 13) often outperform plain Linear Regression on real-world data.


---
## Next Episode

Linear Regression predicts a **number**. But what if the question is yes/no — will this email get opened, will this patient test positive?

That's **Logistic Regression** — coming up next.
